# Movie Recommendation System with DPO
## Overview
This notebook builds a movie recommendation system using Retrieval-Augmented Generation (RAG) and Direct Preference Optimization (DPO).

The pipeline works as follows:
1. Load and clean the Movie dataset
2. Build a semantic retrieval index over movie descriptions and keywords
3. Use a language model to generate recommendation responses
4. Collect user preferences via a Gradio voting interface
5. Fine-tune the model using DPO on collected preferences

## Section 1 — Setup
Install and import all dependencies needed for the full pipeline.

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade torchao
!{sys.executable} -m pip install kagglehub pandas sentence-transformers transformers datasets accelerate trl peft gradio

In [ ]:
import sys
!{sys.executable} -m pip install kagglehub pandas sentence-transformers anthropic gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 29.2 MB/s eta 0:00:00


In [ ]:
import ast
import json
import torch
import pandas as pd
import kagglehub

from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig
from datasets import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cuda


## Section 2 — Data
Load the Movie dataset from Kaggle, clean it, merge with keywords,
and build a combined text field per movie for semantic retrieval.

In [ ]:
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'the-movies-dataset' dataset.
Path to dataset files: /kaggle/input/the-movies-dataset


In [ ]:
movies = pd.read_csv(f"{path}/movies_metadata.csv", low_memory=False)
keywords = pd.read_csv(f"{path}/keywords.csv")

print("movies_metadata shape:", movies.shape)
print("keywords shape:", keywords.shape)

movies_metadata shape: (45466, 24)
keywords shape: (46419, 2)


In [ ]:
keep_cols = [
    'id', 'title', 'overview', 'genres', 'release_date',
    'runtime', 'vote_average'
]

movies = movies[keep_cols]
movies = movies.dropna(subset=['overview', 'title', 'release_date', 'runtime', 'vote_average'])

def parse_genres(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        return [g['name'] for g in genres]
    except:
        return []

def parse_keywords(kw_str):
    try:
        kws = ast.literal_eval(kw_str)
        return [k['name'] for k in kws]
    except:
        return []

movies['genres'] = movies['genres'].apply(parse_genres)
keywords['keywords'] = keywords['keywords'].apply(parse_keywords)

movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)
keywords['id'] = keywords['id'].astype(int)

movies = movies.merge(keywords, on='id', how='left')
movies['keywords'] = movies['keywords'].fillna('').apply(lambda x: x if isinstance(x, list) else [])
movies = movies.drop_duplicates(subset='id').reset_index(drop=True)

print("Clean dataset shape:", movies.shape)

Clean dataset shape: (44405, 8)


In [ ]:
movies = movies.sample(n=5000, random_state=42).reset_index(drop=True)
print('Working with', len(movies), 'movies')

Working with 5000 movies


In [ ]:
def build_text(row):
    genres = ' '.join(row['genres'])
    keywords = ' '.join(row['keywords'])
    overview = row['overview'] if isinstance(row['overview'], str) else ''
    return f"{overview} {genres} {keywords}".strip()

movies['text'] = movies.apply(build_text, axis=1)
print('Text field built.')

Text field built.


## Section 3 — Retrieval
encode each movie's combined text field into an embedding vector using a pretrained sentence transformer. At query time we compute cosine similarity between the user's
description and all movie embeddings to retrieve the most relevant candidates.

In [ ]:
retrieval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
movie_embeddings = retrieval_model.encode(movies['text'].tolist(), convert_to_tensor=True, show_progress_bar=True)

print('Embedding matrix shape:', movie_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding matrix shape: torch.Size([5000, 384])


In [ ]:
def cosine(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

def retrieve_movies(query, top_k=10):
    query_embedding = retrieval_model.encode(query, convert_to_tensor=True)
    scores = [cosine(query_embedding, movie_embeddings[i]) for i in range(len(movies))]
    top_idx = torch.topk(torch.tensor(scores), k=top_k).indices.tolist()
    return [
        {
            'score': scores[idx].item(),
            'movie': movies.iloc[idx][['title', 'overview', 'genres', 'vote_average', 'release_date', 'keywords']].to_dict()
        }
        for idx in top_idx
    ]

## Section 4 — Language Model
load the model and tokenizer directly as AutoModelForCausalLM and AutoTokenizer rather than using a pipeline. This allows us to reuse the same model object for both
generation and DPO training without loading it twice.

In [ ]:
checkpoint = 'HuggingFaceTB/SmolLM2-1.7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype='auto', device_map='auto')

print('Model loaded on:', model.device)

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [ ]:
def generate_response(messages, max_new_tokens=2000, temperature=0.7):
    tokenized = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **tokenized,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_len = tokenized['input_ids'].shape[-1]
    response = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    return response

## Section 5 — Recommendation Pipeline
combine retrieval and generation into a recommendation function.
Given a user description we retrieve the top 5 most semantically similar movies
and prompt the model to pick the two best matches and explain why.

In [ ]:
SYSTEM_MESSAGE = (
    'You are a movie recommendation expert. '
    'You will be given a user request and exactly 5 candidate movies with their overviews. '
    'Choose only the TWO best matches based strictly on the provided overviews and keywords. '
    'Do not use any outside knowledge about the movies. '
    'For each pick, state the title and one sentence explaining why it fits based only on the overview given. '
)

def build_prompt(user_description, retrieved):
    context = ""
    for i, r in enumerate(retrieved, start=1):
        m = r['movie']
        genres = ', '.join(m['genres']) if isinstance(m['genres'], list) else m['genres']
        keywords = ', '.join(m['keywords'][:10]) if isinstance(m['keywords'], list) else m['keywords']
        year = str(m['release_date'])[:4] if pd.notna(m['release_date']) else 'N/A'
        context += (
            f"Movie {i}: {m['title']} ({year}) | Genres: {genres} | Rating: {m['vote_average']}\n"
            f"Keywords: {keywords}\n"
            f"Overview: {m['overview']}\n\n"
        )

    return (
        f"I'm looking for: {user_description}\n\n"
        f"Candidates:\n{context}"
        "Which two movies best match my request and why?"
    )

def generate_two_responses(user_description, top_k=5):
    retrieved = retrieve_movies(user_description, top_k=top_k)
    prompt = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': build_prompt(user_description, retrieved)}
    ]

    response_a = generate_response(prompt, temperature=0.7)
    response_b = generate_response(prompt, temperature=1.2)

    return prompt, response_a, response_b

In [ ]:
# quick sanity check
test_query = "dark psychological thriller with a twist ending"
prompt, response_a, response_b = generate_two_responses(test_query)

print("RESPONSE A:")
print(response_a)
print("\nRESPONSE B:")
print(response_b)

RESPONSE A:
Movie 1: App (2013) and Movie 3: Hellgate (2011)

Movie 1: App (2013) is a dark psychological thriller with a twist ending that fits your request. The film's plot revolves around a young psychology student who becomes involved with a mysterious app that starts terrorizing her, distributing compromising photographs and videos about herself. The app's perfect manipulation of her personal life reveals all her deepest secrets, leading to a shocking twist ending. The movie's darkness and fear, combined with the twist, make it a perfect match for your request.

Movie 3: Hellgate (2011) is a western horror thriller that also fits your request. The film's plot revolves around a businessman, his Thai wife, and their son who experience a horrible accident in Bangkok. They soon find themselves trapped in a world between life and death, where endless darkness lies. The movie is a horror/thriller that is disturbing and engaging, with an unexpected twist ending that changes the point of 

## Section 6 — Synthetic Preference Data
generate preference pairs automatically using query-movie alignment.
For each query we take the highest scoring retrieved movie as `chosen`
and the lowest scoring as `rejected`, then generate an explanation for each.
Pairs are saved to disk so they persist across runtime restarts.

In [ ]:
CHOSEN_SYSTEM = (
    'You are a movie recommendation assistant. '
    'You will be given a user request and exactly 5 candidate movies. '
    'Choose only the TWO best matches based strictly on the provided overviews and keywords. '
    'Do not use any outside knowledge. '
    'For each pick, state only the title and one clear sentence explaining why it fits. '
    'Be concise and specific.'
)

REJECTED_SYSTEM = (
    'You are a movie recommendation assistant. '
    'You will be given a user request and exactly 5 candidate movies. '
    'Choose two movies to recommend.'
)

def generate_synthetic_pair(user_description, top_k=5):
    try:
        retrieved = retrieve_movies(user_description, top_k=top_k)
        user_content = build_prompt(user_description, retrieved)

        chosen_prompt = [
            {'role': 'system', 'content': CHOSEN_SYSTEM},
            {'role': 'user', 'content': user_content}
        ]
        rejected_prompt = [
            {'role': 'system', 'content': REJECTED_SYSTEM},
            {'role': 'user', 'content': user_content}
        ]

        chosen = generate_response(chosen_prompt, temperature=0.5)
        rejected = generate_response(rejected_prompt, temperature=1.3)

        if not chosen or not rejected:
            return None

        return {
            'prompt': chosen_prompt,
            'chosen': [{'role': 'assistant', 'content': chosen}],
            'rejected': [{'role': 'assistant', 'content': rejected}]
        }
    except Exception as e:
        print(f"  Error on '{user_description}': {e}")
        return None

In [ ]:
synthetic_queries = [
    # Drama
    "slow burn drama about family secrets unraveling",
    "courtroom drama with an unexpected verdict",
    "drama about addiction and recovery",
    "drama set in a small rural town with dark secrets",
    "emotional drama about grief and loss",
    "drama about immigration and cultural identity",
    "workplace drama with ethical dilemmas",
    "drama about a failing marriage",
    "drama about an artist struggling for recognition",
    "generational family saga spanning decades",
    # Thriller / Crime
    "serial killer thriller told from the detective's perspective",
    "psychological thriller where the protagonist questions reality",
    "cat and mouse thriller between a cop and a criminal",
    "thriller about a witness to a crime who goes into hiding",
    "financial crime thriller set on wall street",
    "kidnapping thriller with a ticking clock",
    "cold case mystery reopened after new evidence surfaces",
    "thriller about identity theft and paranoia",
    "corporate espionage thriller",
    "thriller set entirely in one location",
    # Horror
    "slow burn atmospheric horror set in an isolated house",
    "horror movie based on folklore and mythology",
    "creature feature horror set in the ocean",
    "psychological horror where the villain is never shown",
    "horror comedy with self aware characters",
    "possession horror with religious themes",
    "found footage horror in an abandoned building",
    "horror about a cursed object passed between victims",
    "vampire horror with a gothic setting",
    "zombie horror focused on survivor dynamics",
    # Comedy
    "dry British comedy about workplace incompetence",
    "road trip comedy with mismatched characters",
    "dark comedy about death and funerals",
    "mockumentary comedy about a small town",
    "comedy about a midlife crisis",
    "buddy cop comedy with unlikely partners",
    "comedy set at a chaotic family reunion",
    "satirical comedy about politics and power",
    "comedy about someone pretending to be something they are not",
    "screwball comedy with fast paced dialogue",
    # Science Fiction
    "hard science fiction about first contact with aliens",
    "time travel science fiction with paradoxes",
    "near future science fiction about surveillance and privacy",
    "science fiction about artificial intelligence gaining consciousness",
    "post apocalyptic science fiction with a lone survivor",
    "science fiction about colonizing a new planet",
    "science fiction thriller set on a space station",
    "science fiction about memory manipulation",
    "biopunk science fiction about genetic engineering",
    "science fiction about parallel universes",
    # Action / Adventure
    "martial arts action movie with stunning choreography",
    "jungle adventure with ancient civilizations",
    "war movie told from both sides of the conflict",
    "revenge thriller with non stop action",
    "spy thriller with globe trotting locations",
    "action movie set during a natural disaster",
    "adventure about a treasure hunt with deadly traps",
    "military action movie with realistic combat",
    "action comedy with over the top set pieces",
    "chase thriller that never slows down",
    # Romance
    "slow burn romance between two rivals",
    "long distance romance with cultural differences",
    "romance set during a historical period",
    "second chance romance between former lovers",
    "romance between two people from different social classes",
    "whimsical romance with magical realism elements",
    "bittersweet romance that doesnt have a happy ending",
    "summer romance between two strangers on vacation",
    "romance that develops through letters or messages",
    "quirky indie romance with unconventional characters",
    # Documentary / Biography
    "documentary about environmental activism",
    "biography about a controversial political figure",
    "documentary about the music industry",
    "biography about a sports legend overcoming adversity",
    "documentary exposing corporate corruption",
    "biography about a scientist who changed the world",
    "documentary about life in an extreme environment",
    "biography about a war hero",
"uplifting story about strangers forming an unexpected friendship",
"intense survival drama in a frozen wilderness",
"coming of age story set in a strict boarding school",
"emotional tale about reconnecting with an estranged parent",
"drama centered around ethical dilemmas in medicine",
"character study of a reclusive genius",
"story about rebuilding life after a public scandal",
"family drama during a holiday gathering gone wrong",
"drama about a whistleblower facing consequences",
"slice of life story about urban loneliness",

"mystery thriller set in a remote mountain village",
"high tension thriller involving a train hijacking",
"thriller about decoding a dangerous secret message",
"political conspiracy thriller with unexpected twists",
"thriller about a journalist uncovering hidden truths",
"crime thriller focused on forensic investigation",
"thriller involving a missing person case with secrets",
"psychological suspense in a luxury hotel",
"thriller about a heist gone wrong",
"undercover operation thriller in a crime syndicate",

"haunted forest horror with eerie atmosphere",
"supernatural horror involving mirrors",
"horror set in a desolate desert town",
"body horror with transformation themes",
"ghost story tied to a tragic past",
"horror about an experiment gone wrong",
"claustrophobic cave exploration horror",
"horror centered on a sinister cult",
"nightmare fueled horror with surreal visuals",
"horror about technology turning against humans",

"lighthearted romantic comedy with awkward encounters",
"comedy about roommates with clashing personalities",
"absurd comedy set in a bizarre workplace",
"fish out of water comedy in a foreign country",
"comedy about planning a disastrous wedding",
"improvisational style comedy with chaotic energy",
"college comedy about unlikely friendships",
"comedy involving mistaken identities",
"holiday themed comedy with family mishaps",
"feel good comedy about starting over",

"space exploration adventure with unknown dangers",
"futuristic dystopia with rebel protagonists",
"science fiction about climate engineering consequences",
"alien invasion from a civilian perspective",
"cyberpunk story about hackers and corporations",
"science fiction with virtual reality immersion",
"story about androids living among humans",
"interstellar journey with existential themes",
"science fiction involving shrinking technology",
"future society controlled by algorithms",

"high speed car chase action movie",
"rescue mission in hostile territory",
"naval battle action at sea",
"mountain climbing adventure with danger",
"action packed prison escape story",
"desert survival adventure with nomads",
"heist story with elaborate planning",
"underwater treasure hunting adventure",
"action story with a reluctant hero",
"battle against overwhelming odds in war",

"romance between coworkers turned rivals",
"unexpected love story during a crisis",
"romance involving a secret identity",
"love story across different timelines",
"romance set in a snowy small town",
"story about falling in love with a best friend",
"romance rekindled at a reunion",
"love story with a dramatic misunderstanding",
"romantic drama with artistic characters",
"romance between a traveler and a local",

"documentary about deep sea exploration",
"biography of an influential entrepreneur",
"documentary on underground art scenes",
"biography of a pioneering inventor",
"documentary about cultural traditions",
"true story of a daring escape",
"documentary about food and culinary history",
"biography of a revolutionary leader",
"documentary exploring space missions",
"biography of an iconic performer",

"dark fantasy with mythical creatures",
"epic quest to restore a lost kingdom",
"fantasy story about magical schools",
"hero journey in a cursed land",
"fantasy adventure with shape shifting beings",
"medieval fantasy war story",
"urban fantasy with hidden magic",
"fantasy about prophecy and destiny",
"magical artifact causing chaos",
"fantasy rebellion against tyranny",

"emotional sports drama about redemption",
"underdog team rising to victory",
"sports biography about perseverance",
"training montage heavy boxing story",
"drama about rivalry between athletes",
"sports story set in high school",
"coach mentoring troubled players",
"story about comeback after injury",
"team bonding overcoming differences",
"sports drama with personal sacrifice",

"noir detective story in a rainy city",
"classic whodunit with multiple suspects",
"mystery involving hidden inheritance",
"investigation into a haunted mansion",
"detective solving crimes in a small town",
"murder mystery during a storm",
"case involving secret societies",
"mystery with unreliable witnesses",
"investigation tied to past crimes",
"mystery unfolding through flashbacks",

"animated adventure with talking animals",
"family friendly story about teamwork",
"animated fantasy with vibrant worlds",
"kids adventure in a magical land",
"heartwarming tale about friendship",
"animation with moral lessons",
"storybook inspired animated journey",
"colorful adventure with whimsical characters",
"family film about overcoming fears",
"animated quest to save a village",

"music driven drama about a struggling band",
"story about rise to fame in entertainment",
"musical with elaborate performances",
"biopic about a groundbreaking artist",
"journey of a street performer",
"music competition story with drama",
"story about rediscovering passion for music",
"band reunion after years apart",
"life of a touring musician",
"music themed romance",

"historical drama set during revolution",
"period piece about royal intrigue",
"story set in ancient civilization",
"historical epic about empire building",
"drama about explorers discovering new lands",
"story about life during wartime rationing",
"historical romance in aristocracy",
"biographical drama set centuries ago",
"story about invention in early industry",
"historical courtroom drama",

"psychological drama about identity crisis",
"story exploring moral ambiguity",
"character driven narrative about obsession",
"introspective film about self discovery",
"drama about isolation in modern society",
"story about coping with trauma",
"exploration of human relationships",
"drama about ambition and downfall",
"philosophical story about purpose",
"narrative about confronting fears",

"thriller involving cyber attacks",
"story about surveillance gone too far",
"tech thriller with rogue programmer",
"mystery around missing digital data",
"AI driven suspense story",
"thriller set in a startup company",
"technology causing unintended chaos",
"digital identity theft thriller",
"online game turning dangerous",
"thriller about hacking conspiracy",

"horror set in a remote island",
"ghost ship horror at sea",
"horror involving cursed family lineage",
"isolated cabin horror in winter",
"horror with hallucinations and paranoia",
"urban legend coming to life",
"creepy hospital horror setting",
"night shift horror in a factory",
"horror with children as central characters",
"supernatural revenge horror",

"romantic comedy about opposites attracting",
"story about love after heartbreak",
"romance in a bustling city",
"unexpected romance during travel",
"love story involving secrets",
"romance between neighbors",
"romantic drama with tragic elements",
"light romance with humorous tone",
"story about finding love later in life",
"romance tested by distance",

"adventure across multiple continents",
"expedition to find lost treasure",
"journey through dangerous terrain",
"explorers facing natural elements",
"adventure involving ancient myths",
"story about survival in jungle",
"trek across desert landscapes",
"expedition with scientific discovery",
"journey through unknown lands",
"adventure with unexpected allies",

"action thriller with rogue agent",
"urban warfare action story",
"action with futuristic weapons",
"hero protecting a key witness",
"explosive mission to stop threat",
"action involving secret government ops",
"battle against organized crime",
"high stakes infiltration mission",
"action set in confined spaces",
"elite team on dangerous assignment",

"documentary about space telescopes",
"real life survival story",
"documentary on wildlife conservation",
"true crime investigation story",
"biography of a famous writer",
"documentary about ancient ruins",
"story about groundbreaking discovery",
"biography of a cultural icon",
"documentary on extreme sports",
"true story of innovation",

"fantasy world with dragons and magic",
"story about enchanted forests",
"magic users hiding in society",
"fantasy about cursed kingdoms",
"epic battle between good and evil",
"journey to destroy a powerful relic",
"fantasy with elemental powers",
"magical realism in everyday life",
"story about mythical guardians",
"fantasy adventure with prophecy",

"comedy about starting a new job",
"awkward humor in social situations",
"comedy about dysfunctional friends",
"satire on modern technology",
"comedy of errors with misunderstandings",
"story about accidental fame",
"comedy involving unusual hobbies",
"family comedy with kids antics",
"comedy about rivalry between neighbors",
"humorous take on serious situations",

"science fiction about time loops",
"future with space colonization conflicts",
"alien species integrating into society",
"science fiction about mind uploading",
"story about interdimensional travel",
"future where robots dominate labor",
"science fiction involving nanotechnology",
"galactic war between factions",
"story about terraforming planets",
"future society with social ranking system",

"drama about rebuilding after disaster",
"story about forgiveness and redemption",
"emotional journey of self acceptance",
"family dealing with sudden change",
"drama about chasing dreams",
"story about betrayal among friends",
"narrative about societal pressure",
"exploration of generational conflict",
"drama about moral choices",
"story about resilience in hardship",

"thriller set during a blackout",
"mystery involving hidden tunnels",
"suspense in a locked building",
"crime story about stolen identity",
"thriller with countdown scenario",
"investigation into strange disappearances",
"story about double lives",
"thriller about secret experiments",
"mystery with coded clues",
"tense standoff situation",

"horror about ancient evil awakening",
"creepy doll horror story",
"nightmare sequences blending reality",
"horror set in abandoned amusement park",
"possession story with modern twist",
"horror with time distortion",
"paranormal investigation gone wrong",
"horror in a remote research station",
"story about cursed land",
"horror with psychological breakdown"
]

In [ ]:
synthetic_data = []

for i, query in enumerate(synthetic_queries):
    print(f"Generating pair {i+1}/{len(synthetic_queries)}: {query}")
    pair = generate_synthetic_pair(query)
    if pair is not None:
        synthetic_data.append(pair)

with open('synthetic_data.json', 'w') as f:
    json.dump(synthetic_data, f)

print(f"\nDone. Total synthetic pairs: {len(synthetic_data)}")
print("Saved to synthetic_data.json")

Generating pair 1/348: slow burn drama about family secrets unraveling
Generating pair 2/348: courtroom drama with an unexpected verdict
Generating pair 3/348: drama about addiction and recovery
Generating pair 4/348: drama set in a small rural town with dark secrets
Generating pair 5/348: emotional drama about grief and loss
Generating pair 6/348: drama about immigration and cultural identity
Generating pair 7/348: workplace drama with ethical dilemmas
Generating pair 8/348: drama about a failing marriage
Generating pair 9/348: drama about an artist struggling for recognition
Generating pair 10/348: generational family saga spanning decades
Generating pair 11/348: serial killer thriller told from the detective's perspective
Generating pair 12/348: psychological thriller where the protagonist questions reality
Generating pair 13/348: cat and mouse thriller between a cop and a criminal
Generating pair 14/348: thriller about a witness to a crime who goes into hiding
Generating pair 15/34

In [ ]:
with open('synthetic_data.json', 'r') as f:
    synthetic_data = json.load(f)

print(f"Loaded {len(synthetic_data)} preference pairs")

Loaded 348 preference pairs


In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = DPOConfig(
    output_dir='movie_dpo_lora',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-7,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    beta=0.5,
    max_length=512,
    fp16=torch.cuda.is_available(),
)

In [ ]:
train_dataset = Dataset.from_list(synthetic_data)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()
print("Training complete.")

Tokenizing train dataset:   0%|          | 0/348 [00:00<?, ? examples/s]

Step,Training Loss
1,0.693147
2,0.693147
3,0.693147
4,0.683483
5,0.693147
6,0.692243
7,0.690122
8,0.693147
9,0.689216
10,0.695592


Training complete.


In [ ]:
# save the fine-tuned model
trainer.save_model('movie_dpo_lora')
tokenizer.save_pretrained('movie_dpo_lora')
print("Model saved to movie_dpo_lora/")

Model saved to movie_dpo_lora/


## Section 7 — Gradio Voting App
A simple interface where users enter a movie description, see two generated responses,
and vote for the better one. Each vote is logged as a DPO preference pair and saved to disk.

In [ ]:
import gradio as gr

voting_data = []

def get_recommendations(user_description):
    retrieved = retrieve_movies(user_description, top_k=5)
    prompt = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': build_prompt(user_description, retrieved)}
    ]

    response_a = generate_response(prompt, temperature=0.7)
    response_b = generate_response(prompt, temperature=1.2)

    candidates = "\n".join([
        f"{i+1}. {r['movie']['title']} ({str(r['movie']['release_date'])[:4]}) | "
        f"Rating: {r['movie']['vote_average']} | "
        f"{', '.join(r['movie']['genres']) if isinstance(r['movie']['genres'], list) else r['movie']['genres']}"
        for i, r in enumerate(retrieved)
    ])

    return candidates, response_a, response_b, prompt, response_a, response_b

def vote(choice, prompt, response_a, response_b):
    chosen = response_a if choice == "A" else response_b
    rejected = response_b if choice == "A" else response_a

    voting_data.append({
        'prompt': prompt,
        'chosen': [{'role': 'assistant', 'content': chosen}],
        'rejected': [{'role': 'assistant', 'content': rejected}]
    })

    with open('voting_data.json', 'w') as f:
        json.dump(voting_data, f)

    return f"Vote logged. Total votes collected: {len(voting_data)}"

with gr.Blocks(title="Movie Recommendation Voter") as app:
    gr.Markdown("# Movie Recommendation Voter")
    gr.Markdown("Enter a movie description, see two responses, and vote for the better one.")

    query_input = gr.Textbox(label="What kind of movie are you looking for?", placeholder="e.g. dark psychological thriller with a twist ending")
    search_btn = gr.Button("Get Recommendations")

    candidates_box = gr.Textbox(label="Retrieved Candidates", lines=6, interactive=False)

    with gr.Row():
        response_a_box = gr.Textbox(label="Response A", lines=8, interactive=False)
        response_b_box = gr.Textbox(label="Response B", lines=8, interactive=False)

    prompt_state = gr.State()
    resp_a_state = gr.State()
    resp_b_state = gr.State()

    with gr.Row():
        vote_a_btn = gr.Button("Vote for A")
        vote_b_btn = gr.Button("Vote for B")

    vote_status = gr.Textbox(label="Status", interactive=False)

    search_btn.click(
        fn=get_recommendations,
        inputs=[query_input],
        outputs=[candidates_box, response_a_box, response_b_box, prompt_state, resp_a_state, resp_b_state]
    )

    vote_a_btn.click(
        fn=lambda p, a, b: vote("A", p, a, b),
        inputs=[prompt_state, resp_a_state, resp_b_state],
        outputs=[vote_status]
    )

    vote_b_btn.click(
        fn=lambda p, a, b: vote("B", p, a, b),
        inputs=[prompt_state, resp_a_state, resp_b_state],
        outputs=[vote_status]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c0cbad8f987dd9f18c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
